In [ ]:
#! pip install scispacy
#! pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_md-0.5.1.tar.gz
#! pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/umls-2020AA-0.5.1.tar.gz


Used Libraries

In [6]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from tqdm import tqdm
from pymongo import MongoClient
import numpy as np

c:\Users\ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧩 Step 1: Setup — Load Preprocessed Dataset

In [7]:
client = MongoClient("mongodb://localhost:27017/")
db = client["Medical_Diagnosis"]

df = pd.DataFrame(list(db["UMLS_Enriched_Illnesses"].find({}, {"_id": 0})))
print("✅ Loaded", len(df), "illness records.")


✅ Loaded 0 illness records.


⚙️ Step 2.3 — Load BioBERT (or ClinicalBERT) Model

You can use BioBERT or ClinicalBERT — both are great.
👉 BioBERT = research articles; ClinicalBERT = hospital notes.
We’ll go with ClinicalBERT first.

In [8]:
# You can also use "dmis-lab/biobert-base-cased-v1.1"
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ Loaded {MODEL_NAME} on {device}")


c:\Users\ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahmed\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

✅ Loaded emilyalsentzer/Bio_ClinicalBERT on cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


⚙️ Step 2.4 — Prepare Text to Embed

We’ll combine all meaningful sections (symptoms, causes, warnings, etc.) into one field — the clinical context.

In [10]:
def combine_text(row):
    text = ""
    for col in ["symptoms", "causes", "warnings", "recommendations"]:
        if col in row and isinstance(row[col], str):
            text += " " + row[col]
    return text.strip()

df["combined_text"] = df.apply(combine_text, axis=1)
print("✅ Combined text fields successfully.")


✅ Combined text fields successfully.


⚙️ Step 2.5 — Generate BioBERT Embeddings

We’ll use the [CLS] token (sentence-level representation).
To save memory, we’ll batch-process and store vectors one by one.

In [11]:
def embed_texts(texts, batch_size=8):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # CLS token
            embeddings.extend(cls_embeddings.cpu().numpy())
    return np.array(embeddings)

df["embedding_vector"] = list(embed_texts(df["combined_text"].tolist()))
print("✅ Generated BioBERT embeddings for all illnesses.")


0it [00:00, ?it/s]

✅ Generated BioBERT embeddings for all illnesses.


In [13]:
print(df.shape)
print(df.head())


(0, 2)
Empty DataFrame
Columns: [combined_text, embedding_vector]
Index: []


In [12]:
collection = db["Illness_Vectors"]
collection.delete_many({})

records = []
for _, row in df.iterrows():
    records.append({
        "illness_name": row["illness_name"],
        "embedding_vector": row["embedding_vector"].tolist(),
        "umls_mappings": row.get("umls_mappings", []),
        "scraped_at": row.get("scraped_at")
    })

collection.insert_many(records)
print(f"✅ Stored {len(records)} embedding records in MongoDB collection 'Illness_Vectors'")


TypeError: documents must be a non-empty list

🩺 Step 2: Named Entity Recognition (NER) — Identify Medical Terms

Use SciSpacy, a biomedical NLP library trained on clinical text.


In [ ]:

nlp = spacy.load("en_core_sci_md")
linker = EntityLinker(resolve_abbreviations=True, name="umls")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "name": "umls"})

def extract_umls_links(text):
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if ent._.umls_ents:
            cui, score = ent._.umls_ents[0]
            concept = linker.umls.cui_to_entity[cui]
            entities.append({
                "text": ent.text,
                "cui": cui,
                "concept_name": concept.canonical_name,
                "definition": concept.definition,
                "score": score
            })
    return entities

df = pd.DataFrame(list(db["Synonym_Expanded_Illnesses"].find({}, {"_id": 0})))
df["umls_mappings"] = df["symptoms"].apply(extract_umls_links)

db["UMLS_Enriched_Illnesses"].delete_many({})
db["UMLS_Enriched_Illnesses"].insert_many(df.to_dict(orient="records"))

print("✅ UMLS linking completed and saved to 'UMLS_Enriched_Illnesses'")


c:\Users\ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ahmed\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
df["umls_mappings"] = df["symptoms"].apply(extract_umls_links)


🧠 Step 3: Medical Synonym & Concept Expansion

We’ll add synonyms and standardized names so your models learn equivalence:

“heart attack” = “myocardial infarction”

“high blood pressure” = “hypertension”

You can use custom rules + optionally UMLS or SNOMED synonyms if available.

Simple Custom Dictionary Example:

# Next Step actually increases recall — your model will recognize equivalent terms.

In [6]:
medical_synonyms = {
    "heart attack": ["myocardial infarction", "cardiac arrest"],
    "flu": ["influenza", "viral infection"],
    "fever": ["high temperature", "pyrexia"],
    "stroke": ["cerebrovascular accident", "brain ischemia"],
    "diabetes": ["high blood sugar", "hyperglycemia"],
}

def expand_synonyms(text):
    for key, syns in medical_synonyms.items():
        for s in syns:
            if key in text:
                text += " " + s
    return text

for col in ["symptoms", "causes", "warnings", "recommendations"]:
    df[col] = df[col].fillna("").apply(expand_synonyms)


🧬 Step 4: Save Enriched Data

In [7]:
enriched_collection = db["Enriched_Illnesses"]
enriched_collection.delete_many({})
enriched_collection.insert_many(df.to_dict(orient="records"))

print("✅ Enriched dataset saved to MongoDB collection: 'Enriched_Illnesses'")


✅ Enriched dataset saved to MongoDB collection: 'Enriched_Illnesses'
